In [1]:
# Imports

from pathlib import Path
from scipy.io import savemat
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import re

In [2]:
# Define participant and set paths

participant = "vp18_LS"

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError: 
    BASE_DIR = Path.cwd().parent
    
INPUT_PATH = BASE_DIR / "analysis" / "participants" / participant / "logs"
OUTPUT_PATH = BASE_DIR / "analysis" / "participants" / participant

os.chdir(INPUT_PATH)

print(BASE_DIR)
print(INPUT_PATH)
print(OUTPUT_PATH)

D:\thesis-matlab-pipeline
D:\thesis-matlab-pipeline\analysis\participants\vp18_LS\logs
D:\thesis-matlab-pipeline\analysis\participants\vp18_LS


In [3]:
# Get logs in dir

logs = [log for log in os.listdir() if log.endswith(".csv")]

print(logs)

['LS1.csv', 'LS2.csv', 'LS3.csv', 'LS4.csv', 'LS_localizer_logging.csv']


In [4]:
def build_arrays(log):
    log_conditions = set(log["condition"].dropna())
    
    if "coherent" in log_conditions:
        condition_order = ["coherent", "incoherent_real", "incoherent_mock", "neutral", "target", "cue"]
        
    elif "Congruent" in log_conditions:
        condition_order = ["Congruent", "Incongruent_Real", "Incongruent_Fake", "Neutral", "True", "cue"]
        
    elif "Face" in log_conditions:
        condition_order = ["Face", "scene", "scrambled"]
        
    else:
        raise KeyError("Invalid log format!")
    
    columns = len(condition_order)

    onset_array = np.empty((columns,), dtype=np.object_)
    duration_array = np.empty((columns,), dtype=np.object_)
    names_array = np.empty((columns,), dtype=np.object_)

    return condition_order, onset_array, duration_array, names_array

In [5]:
def write_conditions_and_onsets(log, condition_order, onset_array):
    is_func = "cue" in condition_order
    
    if is_func:
        onset_name = "_onset" if "Congruent" in condition_order else "_onset_cor"
    else:
        onset_name = "time when it onsets"
    
    for i, condition in enumerate(condition_order):
        if is_func:
            if condition == "cue":
                cond_onsets = log[f"s1{onset_name}"].astype(float)
                
            else:
                cond_onsets = log.loc[
                    log["condition"] == condition, f"s2{onset_name}"
                    ].astype(float)
                
        else:
           cond_onsets = log.loc[
                log["condition"] == condition, onset_name
               ].astype(float)
            
           cond_onsets += 4

        onset_array[i] = cond_onsets.to_numpy()[:, None]

    return onset_array

In [6]:
def write_durations(duration_array, condition_order):
    duration_list = [0.20 for c in condition_order]

    for i, duration in enumerate(duration_list):
        duration_array[i] = duration

    return duration_array

In [7]:
def write_names(condition_order, names_array):

    if "cue" in condition_order:
        condition_order = ["Congruent", "Incongruent_Real", "Incongruent_Fake", "Neutral", "Target", "Cue"] 
    
    for i, name in enumerate(condition_order):
        names_array[i] = name

    return names_array

In [8]:
def to_dict(onset_array, durration_array, names_array):
    mat_dict = {
        "onsets": onset_array,
        "names": names_array,
        "durations": duration_array,
    }

    return mat_dict

In [9]:
def prep_dir(log_id: str):
    match = re.search(r"\d+", log_id)
    
    if match:
        log_number = match.group()
        func_dir = OUTPUT_PATH / f"0{log_number}_func"
    else:
        func_dir = OUTPUT_PATH / "00_loc"
    
    func_dir.mkdir(parents=True, exist_ok=True)

    return func_dir

In [10]:
def save_mcf(output_dir: Path, mat_dict: dict):
    out_dir = output_dir / "mcf"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_file = out_dir / "MCF.mat"
    savemat(out_file, mat_dict)

In [11]:
# Run conversion for all files

for log in logs:
    log_path = INPUT_PATH / log
    df = pd.read_csv(log_path)

    condition_order, empty_onset_array, empty_duration_array, empty_name_array = build_arrays(df)
    
    onset_array = write_conditions_and_onsets(df, condition_order, empty_onset_array)
    duration_array = write_durations(empty_duration_array, condition_order)
    names_array = write_names(condition_order, empty_name_array)
    
    mat_dict = to_dict(onset_array, duration_array, names_array)
    
    mcf_dir = prep_dir(log)
    save_mcf(mcf_dir, mat_dict)
    
    print(f"MCF for {log} saved to {mcf_dir}.")
    for array, values in mat_dict.items():
        print(f"\narray")
        print("================")
        for value in values:
            print(value)

print(f"MCF conversion completed for {logs}.")

MCF for LS1.csv saved to D:\thesis-matlab-pipeline\analysis\participants\vp18_LS\01_func.

array
[[  7.9234842]
 [ 16.4467476]
 [ 58.1287985]
 [ 66.8186596]
 [ 82.2137857]
 [ 89.0189787]
 [133.5365156]
 [155.1028589]
 [173.7671941]
 [236.8656236]
 [280.7325628]
 [305.9852313]
 [346.1992355]
 [353.7385365]
 [361.6278243]
 [376.6726006]
 [383.7948271]
 [425.3431964]
 [453.1979995]
 [460.9205663]
 [468.3262557]
 [475.3649235]]
[[ 42.2329934]
 [ 97.5921733]
 [104.1305867]
 [222.7881316]
 [265.5708779]
 [272.7764451]
 [331.2544732]
 [392.801757 ]
 [401.858616 ]
 [491.260436 ]]
[[ 75.5086453]
 [110.5854431]
 [118.3581564]
 [140.8587359]
 [298.896516 ]
 [314.2581899]
 [322.1307618]
 [338.7437188]
 [368.9667477]
 [418.3545723]
 [443.757332 ]
 [481.9365102]]
[[ 24.9034564]
 [ 34.1270504]
 [ 50.9565179]
 [125.0967156]
 [147.9307611]
 [197.9191276]
 [205.2079621]
 [214.6484312]
 [257.4817155]
 [434.1001021]]
[[164.8270945]
 [182.290552 ]
 [188.5951809]
 [230.2104967]
 [290.5401498]
 [411.0490539]